# Thomson Sampling (TS) based agents

> Agents utelizing the TS based approach for Dynamic pricing and learning problems from https://doi.org/10.48550/arXiv.1604.07463

In [ ]:
#| default_exp agents.dynamic_pricing.TS

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import logging

from abc import ABC, abstractmethod
from typing import Union, Optional, List
import numpy as np
import joblib
import os
import statsmodels.api as sm
from ddopai.agents.dynamic_pricing.utils import GLMLink
from ddopai.envs.base import BaseEnvironment
from ddopai.agents.dynamic_pricing.mushroom_rl import PricingMushroomBaseAgent
from mushroom_rl.core import Agent
from ddopai.utils import MDPInfo
from ddopai.agents.obsprocessors import FlattenTimeDimNumpy
from ddopai.envs.actionprocessors import ClipAction


In [ ]:
#| export
class TSPolicy:
    """
    Minimal Thompson-Sampling agent for the linear-demand model
        D = x⊤α + p · x⊤β + ε
    – Gaussian prior  θ∼N(0,λ⁻¹I)
    – Incremental ridge update of  M_t = λI + Σ z zᵀ  and  θ̂_t = M_t⁻¹ q_t
    – One Gaussian posterior draw at each round, priced by  p* = -a / (2b)
    """

    # ---------- ctor ---------------------------------------------------------
    def __init__(self,
                 lam: float,
                 environment_info: MDPInfo,
                 price_function,               # takes (x, a, b) ➜ price
                 actionprocessors=None,
                 ex_prices=None,
                 init_scale=None):
        """
        lam      : ridge / prior precision λ
        price_function(x, a, b) returns the quadratic-optimal price (usually -a/2b)
        ex_prices : iterable of k ∈{0,1,2,…} initial prices; can be empty
        init_scale        : exploration std-multiplier; default √d / 25
        """

        d_feat = environment_info.observation_space['features'].shape[0]
        self.d_param  = 2 * d_feat
        self.lam      = lam
        self.scale    = init_scale or (np.sqrt(d_feat) / 25.0)

        # incremental posterior
        self.M_inv = np.eye(self.d_param) / lam   if lam else np.eye(self.d_param)
        self.q     = np.zeros(self.d_param)

        # current point estimate
        self.alpha = np.zeros(d_feat)
        self.beta  = np.zeros(d_feat)

        # misc
        self.env_info   = environment_info
        self.price_fn   = price_function
        self.t          = 0
        self.ex_prices     = np.asarray(ex_prices) if ex_prices is not None else np.empty(0)

        # processors (only clip)
        self.actionprocessors = actionprocessors or []
        self.actionprocessors.append(
            ClipAction(environment_info.action_space.low,
                       environment_info.action_space.high)
        )

    # ---------- draw_action ---------------------------------------------------
    def draw_action(self, observation):
        x = observation['features']

        # warm-start if required
        if self.t < self.ex_prices.size:
            p = self.ex_prices[self.t]
        else:
            # posterior sample
            L    = np.linalg.cholesky(self.M_inv)
            noise    = np.random.randn(self.d_param)
            theta_hat    = self.M_inv @ self.q + self.scale * (L @ noise)

            a    = x @ theta_hat[:x.size]
            b    = x @ theta_hat[x.size:]
            b = np.minimum(np.array([-0.01]), b)
            a = np.maximum(np.array([0.01]), a)

            p    = self.price_fn(np.ones_like(a), a, b)         # usually -a / (2b)

        for proc in self.actionprocessors:
            p = proc(p)
        return np.array(p, dtype=np.float32)

    # ---------- fit -----------------------------------------------------------
    def fit(self, X, D, price):
        """Update posterior with (x,p,D)."""
        z = np.concatenate([X, X * price])         # length 2d
        Mz = self.M_inv @ z
        self.M_inv -= np.outer(Mz, Mz) / (1.0 + z @ Mz)
        self.q     += z * float(D)

        θ_hat = self.M_inv @ self.q
        split = θ_hat.size // 2
        self.alpha, self.beta = θ_hat[:split], θ_hat[split:]

        self.t += 1

    # ---------- helpers -------------------------------------------------------
    def reset(self):
        pass

    def update_task(self, env):
        """Start fresh on a new MDP / feature dimension."""
        self.environment_info = env.mdp_info
        self.d = self.environment_info.observation_space['features'].shape[0] * 2
        self.M_inv = np.eye(self.d) / self.lam   if self.lam != 0 else np.eye(self.d)
        self.q = np.zeros(self.d)
        self.actionprocessors[-1] = ClipAction(self.environment_info.action_space.low, self.environment_info.action_space.high)
        self.t = 0


In [ ]:
#| export
class TSCoreAgent(Agent):
    """
    Base class for TS agents.
    """

    def __init__(self,
                 lam: float,
                 environment_info: MDPInfo,
                 agent_name: str | None = None,
                 price_function=None,               # takes (x, a, b) ➜ price
                 actionprocessors=None,
                 ex_prices=None,
                 init_scale=None):
        
        policy = TSPolicy(lam=lam, environment_info=environment_info, actionprocessors=actionprocessors, ex_prices=ex_prices, price_function=price_function, init_scale=init_scale)
        self.agent_name = "TS"
        super().__init__(environment_info, policy)
        
    def fit(self, dataset, **kwargs):
        X = dataset[0][0]['features']
        Y = kwargs["demand"][0]
        action = dataset[0][1]
        self.policy.fit(X, Y, action)
    def update_task(self, env):
        self.policy.update_task(env)

In [ ]:
#| export
class TSAgent(PricingMushroomBaseAgent):
    """
    Wrapper class for TSCoreAgent to interact with MushroomRL.
    """
    def __init__(self,
                 lam: float,
                 environment_info: MDPInfo,
                 obsprocessors: Optional[List[object]] =[],
                 actionprocessors: Optional[List[object]] = [],
                 agent_name: str | None = None,
                 ex_prices: np.ndarray | None = None,
                 price_function = None,
                 g = None,
                 ):
        self.agent = TSCoreAgent(lam=lam, environment_info=environment_info,
                                 actionprocessors=actionprocessors, 
                                 agent_name=agent_name, 
                                 ex_prices=ex_prices, 
                                 price_function=price_function, 
                                 )
        super().__init__(environment_info=environment_info, obsprocessors=obsprocessors, agent_name=agent_name)
    def update_task(self, env: object):
        """ Update the environment specific parameters of the agent """
        self.agent.update_task(env)